In [1]:
from pyspark.sql.functions import avg, count, round

In [ ]:
catalog = dbutils.widgets.get("catalog")

In [ ]:
df = spark.read.table(f"{catalog}.02_silver.jc_citibike")

In [3]:
df.show()

+----------------+---------------+--------------------+--------------------+------------------+--------------------+------------------+--------------------+
|         ride_id|trip_start_date|          started_at|            ended_at|start_station_name|    end_station_name|trip_duration_mins|            metadata|
+----------------+---------------+--------------------+--------------------+------------------+--------------------+------------------+--------------------+
|29DAF43DD84B4B7A|     2025-03-20|2025-03-20 18:58:...|2025-03-20 19:00:...|   6 St & Grand St|Mama Johnson Fiel...|              2.25|{pipeline_id -> p...|
|B11B4220F7195025|     2025-03-29|2025-03-29 11:01:...|2025-03-29 11:11:...|  Heights Elevator|        Jersey & 3rd| 9.733333333333333|{pipeline_id -> p...|
|18D5B30305F602B9|     2025-03-01|2025-03-01 16:05:...|2025-03-01 16:07:...|      Jersey & 3rd|       Hamilton Park| 2.183333333333333|{pipeline_id -> p...|
|532EB2D9DB68567D|     2025-03-21|2025-03-21 18:44:...|202

In [4]:

df = df.\
    groupBy("trip_start_date", "start_station_name").\
    agg(
    round(avg("trip_duration_mins"),2).alias("avg_trip_duration_mins"),
    count("ride_id").alias("total_trips")
    )

In [5]:
df.show()

+---------------+--------------------+----------------------+-----------+
|trip_start_date|  start_station_name|avg_trip_duration_mins|total_trips|
+---------------+--------------------+----------------------+-----------+
|     2025-03-21|11 St & Washingto...|                  5.79|         49|
|     2025-03-28|        Brunswick St|                  7.93|         44|
|     2025-03-21|        Brunswick St|                 46.84|         27|
|     2025-03-12|        Newport PATH|                  6.63|         61|
|     2025-03-25|Southwest Park - ...|                  5.89|         25|
|     2025-03-27|South Waterfront ...|                  7.64|         59|
|     2025-03-27|    Marin Light Rail|                  6.45|         41|
|     2025-03-24|     4 St & River St|                  7.54|         27|
|     2025-03-08|Hoboken Ave at Mo...|                  7.19|         39|
|     2025-03-04|         Astor Place|                  6.87|         13|
|     2025-03-31|          Bergen Ave|

In [ ]:
df.write.\
    mode("overwrite").\
    option("overwriteSchema", "true").\
    saveAsTable(f"{catalog}.03_gold.daily_station_performance")